In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

cleaned_file = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "yellow_tripdata_2024-01_cleaned.parquet"
)

cleaned_df = pd.read_parquet(cleaned_file)

print("Shape:", cleaned_df.shape)

print("\nColumns:")
print(cleaned_df.columns.tolist())

Shape: (2964568, 35)

Columns:
['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_forward_flag', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'trip_duration_minutes', 'pickup_date', 'pickup_year', 'pickup_month', 'pickup_day', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'avg_speed_mph', 'is_negative_duration', 'is_zero_duration', 'is_negative_distance', 'is_zero_distance', 'is_negative_fare', 'is_negative_total', 'is_high_speed']


In [2]:
quality_flag_columns = [
    "is_negative_duration",
    "is_zero_duration",
    "is_negative_distance",
    "is_zero_distance",
    "is_negative_fare",
    "is_negative_total",
    "is_high_speed"
]

cleaned_df[quality_flag_columns].sum()

is_negative_duration        0
is_zero_duration          814
is_negative_distance        0
is_zero_distance        60371
is_negative_fare        37448
is_negative_total       35504
is_high_speed            1024
dtype: int64

In [3]:
SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "summary"
    / "cleaning_summary.csv"
)

cleaning_summary = pd.read_csv(SUMMARY_FILE)

cleaning_summary.head()

,source_file,taxi_type,rows_before_cleaning,rows_after_cleaning,rows_removed,negative_duration,zero_duration,negative_distance,zero_distance,negative_fare,negative_total,high_speed,invalid_timestamp,output_file
0,yellow_tripdata_2024-01.parquet,yellow,2964624,2964568,56,56,814,0,60371,37448,35504,1024,15,D:\Personal\MineProjects\PowerBIProjects\Proje...
1,yellow_tripdata_2024-02.parquet,yellow,3007526,3007519,7,7,796,0,68225,40655,36080,883,2,D:\Personal\MineProjects\PowerBIProjects\Proje...
2,yellow_tripdata_2024-03.parquet,yellow,3582628,3582511,117,117,1011,0,87168,58464,44311,1178,2,D:\Personal\MineProjects\PowerBIProjects\Proje...
3,yellow_tripdata_2024-04.parquet,yellow,3514289,3514219,70,70,1019,0,45813,57568,43749,1062,4,D:\Personal\MineProjects\PowerBIProjects\Proje...
4,yellow_tripdata_2024-05.parquet,yellow,3723833,3723791,42,42,1105,0,49791,59947,48992,1071,7,D:\Personal\MineProjects\PowerBIProjects\Proje...


In [5]:
print("Total files:", len(cleaning_summary))

print("\nOriginal rows:", cleaning_summary["rows_before_cleaning"].sum())
print("Cleaned rows:", cleaning_summary["rows_after_cleaning"].sum())
print("Rows removed:", cleaning_summary["rows_removed"].sum())

print("\nNegative duration:", cleaning_summary["negative_duration"].sum())
print("Zero duration:", cleaning_summary["zero_duration"].sum())
print("Negative distance:", cleaning_summary["negative_distance"].sum())
print("Zero distance:", cleaning_summary["zero_distance"].sum())
print("Negative fare:", cleaning_summary["negative_fare"].sum())
print("Negative total:", cleaning_summary["negative_total"].sum())
print("High speed:", cleaning_summary["high_speed"].sum())
print("Invalid timestamp:", cleaning_summary["invalid_timestamp"].sum())

Total files: 62

Original rows: 117806981
Cleaned rows: 117801813
Rows removed: 5168

Negative duration: 5168
Zero duration: 885926
Negative distance: 0
Zero distance: 3081859
Negative fare: 3727688
Negative total: 1735219
High speed: 36258
Invalid timestamp: 2279


In [6]:
check = (
    cleaning_summary["rows_before_cleaning"]
    - cleaning_summary["rows_after_cleaning"]
)

print(
    "Row-count reconciliation:",
    check.sum() == cleaning_summary["rows_removed"].sum()
)

Row-count reconciliation: True


In [7]:
taxi_summary = (
    cleaning_summary
    .groupby("taxi_type")
    .agg(
        files=("source_file", "count"),
        rows_before=("rows_before_cleaning", "sum"),
        rows_after=("rows_after_cleaning", "sum"),
        rows_removed=("rows_removed", "sum"),
        invalid_timestamps=("invalid_timestamp", "sum")
    )
    .reset_index()
)

taxi_summary

,taxi_type,files,rows_before,rows_after,rows_removed,invalid_timestamps
0,green,31,1548020,1546669,1351,39
1,yellow,31,116258961,116255144,3817,2240


In [8]:
year_summary = (
    cleaning_summary
    .assign(
        year=cleaning_summary["source_file"].str.extract(
            r"_(\d{4})-"
        )[0].astype(int)
    )
    .groupby(["taxi_type", "year"])
    .agg(
        files=("source_file", "count"),
        rows_before=("rows_before_cleaning", "sum"),
        rows_after=("rows_after_cleaning", "sum"),
        rows_removed=("rows_removed", "sum"),
        invalid_timestamps=("invalid_timestamp", "sum")
    )
    .reset_index()
)

year_summary

,taxi_type,year,files,rows_before,rows_after,rows_removed,invalid_timestamps
0,green,2024,12,660218,660216,2,9
1,green,2025,12,591375,590031,1344,1
2,green,2026,7,296427,296422,5,29
3,yellow,2024,12,41169720,41168145,1575,49
4,yellow,2025,12,48722602,48720367,2235,8
5,yellow,2026,7,26366639,26366632,7,2183


In [9]:
timestamp_flags = cleaning_summary[
    cleaning_summary["invalid_timestamp"] > 0
][
    [
        "source_file",
        "taxi_type",
        "invalid_timestamp"
    ]
].copy()

timestamp_flags.sort_values(
    "invalid_timestamp",
    ascending=False
)

,source_file,taxi_type,invalid_timestamp
30,yellow_tripdata_2026-07.parquet,yellow,2174
61,green_tripdata_2026-07.parquet,green,26
0,yellow_tripdata_2024-01.parquet,yellow,15
4,yellow_tripdata_2024-05.parquet,yellow,7
27,yellow_tripdata_2026-04.parquet,yellow,4
5,yellow_tripdata_2024-06.parquet,yellow,4
3,yellow_tripdata_2024-04.parquet,yellow,4
11,yellow_tripdata_2024-12.parquet,yellow,3
6,yellow_tripdata_2024-07.parquet,yellow,3
10,yellow_tripdata_2024-11.parquet,yellow,3


In [11]:
yellow_july = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "yellow"
    / "2026"
    / "yellow_tripdata_2026-07.parquet",
    columns=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "DOLocationID"
    ]
)

yellow_july["tpep_pickup_datetime"] = pd.to_datetime(
    yellow_july["tpep_pickup_datetime"]
)

yellow_july["tpep_dropoff_datetime"] = pd.to_datetime(
    yellow_july["tpep_dropoff_datetime"]
)

invalid_july = yellow_july[
    (yellow_july["tpep_pickup_datetime"] < "2024-01-01")
    | (yellow_july["tpep_pickup_datetime"] > "2026-07-31 23:59:59")
    | (yellow_july["tpep_dropoff_datetime"] < "2024-01-01")
    | (yellow_july["tpep_dropoff_datetime"] > "2026-07-31 23:59:59")
    | yellow_july["tpep_pickup_datetime"].isna()
    | yellow_july["tpep_dropoff_datetime"].isna()
]

print("Invalid July records:", len(invalid_july))

invalid_july.head(20)

Invalid July records: 2174


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
2176020,2008-12-30 23:06:00,2008-12-30 23:15:45,1.65,11.4,17.90,43,141
2388824,2026-07-30 08:52:12,2026-08-01 09:04:23,0.00,3.0,7.00,143,143
2428934,2026-08-01 12:23:56,2026-08-01 12:33:55,1.40,11.4,18.90,239,236
2428935,2026-08-01 12:44:06,2026-08-01 13:06:32,2.99,21.2,34.14,161,107
2428936,2026-08-01 13:26:43,2026-08-01 13:36:35,1.17,10.7,19.06,114,107
2428937,2026-08-01 13:40:27,2026-08-01 13:57:31,2.20,17.0,26.67,107,229
2428938,2026-08-01 14:21:09,2026-08-01 15:07:46,10.98,57.6,92.77,48,138
2428939,2026-08-01 16:51:09,2026-08-01 17:06:11,7.29,28.9,39.90,138,98
2428940,2026-08-01 19:50:49,2026-08-01 20:11:36,13.37,50.6,66.60,138,265
2428941,2026-08-02 00:15:12,2026-08-02 00:44:03,10.54,42.9,48.90,132,36


In [12]:
invalid_july[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "DOLocationID"
    ]
].sort_values("tpep_pickup_datetime").head(30)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
2176020,2008-12-30 23:06:00,2008-12-30 23:15:45,1.65,11.40,17.90,43,141
2388824,2026-07-30 08:52:12,2026-08-01 09:04:23,0.00,3.00,7.00,143,143
2474214,2026-07-31 00:19:40,2026-08-01 00:15:00,2.10,14.90,20.65,164,114
2489990,2026-07-31 09:00:37,2026-08-01 20:06:49,10.04,33.80,37.30,132,82
2487228,2026-07-31 09:40:43,2026-08-01 09:38:31,3.17,19.10,26.23,107,140
2487140,2026-07-31 09:44:44,2026-08-01 09:44:12,3.76,24.00,34.50,186,238
2495857,2026-07-31 10:58:17,2026-08-01 00:00:00,0.82,7.90,12.65,237,161
2499318,2026-07-31 12:25:27,2026-08-01 12:20:25,0.65,7.90,12.65,237,163
2501501,2026-07-31 12:40:08,2026-08-01 00:00:00,14.08,70.00,82.21,219,230
2522438,2026-07-31 14:48:30,2026-08-01 14:05:26,2.44,17.00,29.10,162,148


In [13]:
print("Pickup range:")
print(invalid_july["tpep_pickup_datetime"].min())
print(invalid_july["tpep_pickup_datetime"].max())

print("\nDropoff range:")
print(invalid_july["tpep_dropoff_datetime"].min())
print(invalid_july["tpep_dropoff_datetime"].max())

Pickup range:
2008-12-30 23:06:00
2026-08-05 20:54:00

Dropoff range:
2008-12-30 23:15:45
2026-08-05 21:04:08


In [14]:
july_old = invalid_july[
    invalid_july["tpep_pickup_datetime"] < "2024-01-01"
]

july_future_pickup = invalid_july[
    invalid_july["tpep_pickup_datetime"] > "2026-07-31 23:59:59"
]

july_cross_month = invalid_july[
    (invalid_july["tpep_pickup_datetime"] >= "2026-07-01")
    & (invalid_july["tpep_pickup_datetime"] <= "2026-07-31 23:59:59")
    & (invalid_july["tpep_dropoff_datetime"] > "2026-07-31 23:59:59")
]

print("Old pickup records:", len(july_old))
print("Future pickup records:", len(july_future_pickup))
print("July pickup + August dropoff:", len(july_cross_month))

Old pickup records: 1
Future pickup records: 38
July pickup + August dropoff: 2135


In [15]:
print(
    invalid_july[
        invalid_july["tpep_pickup_datetime"] < "2024-01-01"
    ][
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "fare_amount",
            "total_amount"
        ]
    ].head(10)
)

        tpep_pickup_datetime tpep_dropoff_datetime  trip_distance  \
2176020  2008-12-30 23:06:00   2008-12-30 23:15:45           1.65   

         fare_amount  total_amount  
2176020         11.4          17.9  


## Timestamp Quality Rules

The project covers trips from January 2024 through July 2026.

The pickup datetime is used as the primary temporal reference because the trip's reporting period is determined by when the trip begins.

### Findings

The full dataset contains 117,806,981 trip records.

Initial timestamp analysis identified:

- 5,168 records with negative trip duration.
- 885,926 records with zero trip duration.
- 824 records with pickup timestamps before their source-file month.
- 437 records with pickup timestamps after their source-file month.

Additional investigation of July 2026 Yellow Taxi data identified records with August dropoff timestamps. These are valid cross-month trips when the pickup occurred on July 31.

Therefore, a dropoff timestamp after the analytical period does not automatically make a trip invalid.

### Final Rules

1. `pickup_datetime` determines the analytical reporting period.
2. Trips with pickup dates from January 2024 through July 2026 are within scope.
3. Trips with pickup dates outside this period are flagged as outside the analytical scope.
4. A July 31 trip that ends on August 1 is retained.
5. Negative-duration records are removed because the temporal relationship between pickup and dropoff is invalid.
6. Zero-duration records are retained.
7. Original timestamp values are never artificially corrected.
